<a href="https://colab.research.google.com/github/Duszed/Nomad-Aerospace-Flight-Systems/blob/main/notebooks/mission_planner_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import numpy as np
import json
from IPython.display import HTML, display

# 1. EXACT PARCEL CORNER COORDINATES (Syrdarya Farm)
P_NW = [40.41098304649329, 68.84130047050098]  # Home Base / Entry
P_NE = [40.411748029633785, 68.8427874163686]  # North-East Edge
P_SE = [40.41094632708382, 68.84521475502818]  # South-East Corner
P_SW = [40.40970397525738, 68.84299639254459]  # South-West Corner

field_boundary = [P_NW, P_NE, P_SE, P_SW, P_NW]

# 2. DISTANCE HELPER & 6M SWATH PASSES
def get_distance_m(p1, p2):
    dlat = (p2[0] - p1[0]) * 111139.0
    dlon = (p2[1] - p1[1]) * 111139.0 * np.cos(np.radians(p1[0]))
    return np.hypot(dlat, dlon)

left_edge_m = get_distance_m(P_NW, P_SW)
right_edge_m = get_distance_m(P_NE, P_SE)
avg_length_m = (left_edge_m + right_edge_m) / 2.0
swath_width_m = 6.0
num_passes = int(np.ceil(avg_length_m / swath_width_m))

# 3. GENERATE BOUSTROPHEDON FLIGHT PATH
flight_path = []
for i in range(num_passes + 1):
    t = i / float(num_passes)
    pt_left = [P_NW[0] * (1.0 - t) + P_SW[0] * t, P_NW[1] * (1.0 - t) + P_SW[1] * t]
    pt_right = [P_NE[0] * (1.0 - t) + P_SE[0] * t, P_NE[1] * (1.0 - t) + P_SE[1] * t]

    if i % 2 == 0:
        flight_path.extend([pt_left, pt_right])
    else:
        flight_path.extend([pt_right, pt_left])

mission_distance_m = sum(get_distance_m(flight_path[j], flight_path[j+1]) for j in range(len(flight_path)-1))

# 4. AVIONICS SIMULATOR INTERFACE (CONSOLIDATED HOME & REFILL)
html_code = f"""
<!DOCTYPE html>
<html>
<head>
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
  <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
  <style>
    body {{ margin: 0; padding: 0; background: #0b0f19; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, monospace; }}
    #sim-wrap {{ position: relative; width: 100%; height: 600px; overflow: hidden; }}
    #map {{ width: 100%; height: 100%; background: #1a1a1a; }}

    .hud-box {{
      position: absolute; top: 12px; right: 12px; z-index: 2000;
      background: rgba(10, 15, 25, 0.94); border: 1px solid #00e5ff;
      border-radius: 6px; padding: 12px 16px; color: #fff; font-size: 12px;
      min-width: 210px; pointer-events: auto;
      box-shadow: 0 4px 15px rgba(0,0,0,0.6);
    }}
    .hud-title {{ font-size: 11px; text-transform: uppercase; color: #00e5ff; font-weight: bold; margin-bottom: 6px; border-bottom: 1px solid rgba(0,229,255,0.2); padding-bottom: 4px; }}
    .hud-row {{ display: flex; justify-content: space-between; margin-bottom: 4px; }}
    .hud-val {{ font-weight: bold; color: #00ff66; }}

    .ctrl-box {{
      position: absolute; bottom: 18px; left: 50%; transform: translateX(-50%); z-index: 2000;
      background: rgba(15, 20, 30, 0.96); border: 1px solid rgba(255,255,255,0.25);
      border-radius: 20px; padding: 10px 18px; display: flex; gap: 10px; align-items: center;
      pointer-events: auto; box-shadow: 0 4px 25px rgba(0,0,0,0.7); flex-wrap: wrap; justify-content: center;
    }}
    .sim-btn {{
      background: #00e5ff; color: #000; border: none; font-weight: bold;
      padding: 7px 15px; border-radius: 14px; cursor: pointer; font-size: 12px;
      transition: all 0.15s ease;
    }}
    .sim-btn:hover {{ opacity: 0.9; transform: scale(1.02); }}
    .btn-warn {{ background: #ffaa00; color: #000; }}
    .btn-danger {{ background: #ff3333; color: #fff; }}
    .btn-sec {{ background: rgba(255,255,255,0.1); color: #fff; border: 1px solid rgba(255,255,255,0.2); }}

    .slider-container {{
      display: flex; align-items: center; gap: 8px; color: #aaa; font-size: 11px;
      border-left: 1px solid rgba(255,255,255,0.2); padding-left: 10px;
    }}
    input[type=range] {{
      accent-color: #00e5ff; cursor: pointer; width: 85px;
    }}
  </style>
</head>
<body>
<div id="sim-wrap">
  <div id="map"></div>

  <div id="hud" class="hud-box">
    <div class="hud-title">NOMAD K30 AVIONICS HUD</div>
    <div class="hud-row"><span>STATUS:</span><span id="hud-status" class="hud-val" style="color:#ffcc00">READY</span></div>
    <div class="hud-row"><span>WAYPOINT:</span><span id="hud-wp" class="hud-val">0 / {len(flight_path)}</span></div>
    <div class="hud-row"><span>ALTITUDE:</span><span id="hud-alt" class="hud-val">0.0 m</span></div>
    <div class="hud-row"><span>AIRSPEED:</span><span id="hud-spd" class="hud-val">0.0 m/s</span></div>
    <div class="hud-row"><span>PUMP:</span><span id="hud-pump" class="hud-val" style="color:#888">OFF</span></div>
    <div class="hud-row"><span>TANK (30L):</span><span id="hud-tank" class="hud-val" style="color:#00e5ff">30.0 L</span></div>
    <div class="hud-row"><span>BATTERY (14S):</span><span id="hud-bat" class="hud-val" style="color:#00ff66">100%</span></div>
  </div>

  <div id="controls" class="ctrl-box">
    <button id="playBtn" class="sim-btn" onclick="window.toggleFlight()">▶ START</button>
    <button id="homeBtn" class="sim-btn btn-warn" onclick="window.triggerHome()">🏠 RETURN HOME / REFILL</button>
    <button class="sim-btn btn-danger" onclick="window.triggerEmgLand()">🛑 EMG LAND</button>
    <button class="sim-btn btn-sec" onclick="window.resetFlight()">↺ RESET</button>

    <div class="slider-container">
      <span>SPEED:</span>
      <input type="range" id="speedSlider" min="0.5" max="8.0" step="0.5" value="2.0" oninput="window.setSpeed(this.value)">
      <span id="speedVal" style="color:#00e5ff; font-weight:bold; min-width: 28px;">2.0x</span>
    </div>
  </div>
</div>

<script>
  const waypoints = {json.dumps(flight_path)};
  const boundary = {json.dumps(field_boundary)};
  const HOME_POS = waypoints[0];

  // 1. Map Initialization
  const map = L.map('map', {{ zoomControl: true, attributionControl: false }});
  L.tileLayer('https://mt1.google.com/vt/lyrs=y&x={{x}}&y={{y}}&z={{z}}', {{ maxZoom: 20 }}).addTo(map);

  ['hud', 'controls'].forEach(id => {{
    const el = document.getElementById(id);
    L.DomEvent.disableClickPropagation(el);
    L.DomEvent.disableScrollPropagation(el);
  }});

  const poly = L.polygon(boundary, {{ color: '#ff3333', weight: 3, fillOpacity: 0.15 }}).addTo(map);
  map.fitBounds(poly.getBounds(), {{ padding: [30, 30] }});

  L.polyline(waypoints, {{ color: '#00e5ff', weight: 1.5, opacity: 0.35, dashArray: '3, 4' }}).addTo(map);
  const sprayTrail = L.polyline([], {{ color: '#00ff66', weight: 4, opacity: 0.85 }}).addTo(map);

  // 2. 4-Arm Industrial Quadcopter Frame
  const quadSvg = `
    <div id="drone-icon-rot" style="width:36px; height:36px; transform-origin:center; filter: drop-shadow(0 0 6px #00e5ff);">
      <svg viewBox="0 0 100 100" width="36" height="36">
        <line x1="50" y1="50" x2="18" y2="18" stroke="#fff" stroke-width="6" stroke-linecap="round"/>
        <line x1="50" y1="50" x2="82" y2="18" stroke="#fff" stroke-width="6" stroke-linecap="round"/>
        <line x1="50" y1="50" x2="18" y2="82" stroke="#fff" stroke-width="6" stroke-linecap="round"/>
        <line x1="50" y1="50" x2="82" y2="82" stroke="#fff" stroke-width="6" stroke-linecap="round"/>
        <circle cx="50" cy="50" r="16" fill="#111" stroke="#00e5ff" stroke-width="5"/>
        <circle cx="18" cy="18" r="9" fill="#00e5ff" stroke="#fff" stroke-width="2"/>
        <circle cx="82" cy="18" r="9" fill="#00e5ff" stroke="#fff" stroke-width="2"/>
        <circle cx="18" cy="82" r="9" fill="#00e5ff" stroke="#fff" stroke-width="2"/>
        <circle cx="82" cy="82" r="9" fill="#00e5ff" stroke="#fff" stroke-width="2"/>
        <polygon points="50,30 42,46 58,46" fill="#ff3333"/>
      </svg>
    </div>`;

  const droneMarker = L.marker(HOME_POS, {{
    icon: L.divIcon({{ html: quadSvg, className: '', iconSize: [36, 36], iconAnchor: [18, 18] }})
  }}).addTo(map);

  setTimeout(() => {{ map.invalidateSize(); }}, 300);
  setTimeout(() => {{ map.invalidateSize(); }}, 800);

  // 3. Flight Engine Dynamics
  let mode = 'STANDBY'; // 'STANDBY', 'SPRAYING', 'PAUSED', 'RTH', 'RESUMING', 'EMG_LAND', 'FINISHED'
  let wpIdx = 0;
  let t = 0.0;
  let speedMult = 2.0;
  const BASE_SWATH_STEP = 0.005;
  const BASE_TRANSIT_RATE = 0.000007; // Natural 6.5 m/s transit pace in degrees
  let animId = null;

  let currentPos = [...HOME_POS];
  let tank = 30.0;
  let battery = 100.0;
  let altitude = 0.0;
  let breakPoint = null;

  function calcBearing(a, b) {{
    const dLon = (b[1] - a[1]) * Math.PI / 180;
    const lat1 = a[0] * Math.PI / 180;
    const lat2 = b[0] * Math.PI / 180;
    const y = Math.sin(dLon) * Math.cos(lat2);
    const x = Math.cos(lat1) * Math.sin(lat2) - Math.sin(lat1) * Math.cos(dLon);
    return (Math.atan2(y, x) * 180 / Math.PI + 360) % 360;
  }}

  function updateHUD(status, statusColor, wpText, alt, spd, pumpActive) {{
    const sEl = document.getElementById('hud-status');
    sEl.innerText = status;
    sEl.style.color = statusColor;
    document.getElementById('hud-wp').innerText = wpText;
    document.getElementById('hud-alt').innerText = alt.toFixed(1) + ' m';
    document.getElementById('hud-spd').innerText = spd.toFixed(1) + ' m/s';

    const pEl = document.getElementById('hud-pump');
    pEl.innerText = pumpActive ? 'ACTIVE (1.2L/m)' : 'OFF';
    pEl.style.color = pumpActive ? '#00ff66' : '#888';

    document.getElementById('hud-tank').innerText = tank.toFixed(1) + ' L';
    const bEl = document.getElementById('hud-bat');
    bEl.innerText = Math.round(battery) + '%';
    bEl.style.color = battery > 25 ? '#00ff66' : '#ff3333';
  }}

  function step() {{
    const swathStep = BASE_SWATH_STEP * speedMult;
    const transitStep = BASE_TRANSIT_RATE * speedMult;

    // --- 1. REGULAR SPRAY SWATH RUN ---
    if (mode === 'SPRAYING') {{
      altitude = Math.min(3.0, altitude + 0.1);

      if (wpIdx >= waypoints.length - 1) {{
        mode = 'FINISHED';
        document.getElementById('playBtn').innerText = '↺ REPLAY';
        updateHUD('MISSION COMPLETE', '#00ff66', `${{waypoints.length}} / ${{waypoints.length}}`, 0.0, 0.0, false);
        return;
      }}

      const p0 = waypoints[wpIdx];
      const p1 = waypoints[wpIdx + 1];
      t += swathStep;

      if (t >= 1.0) {{
        t = 0.0;
        wpIdx++;
        tank = Math.max(0, tank - (30.0 / waypoints.length));
        battery = Math.max(5, battery - (80.0 / waypoints.length));
      }}

      currentPos = [p0[0] + (p1[0] - p0[0]) * t, p0[1] + (p1[1] - p0[1]) * t];
      droneMarker.setLatLng(currentPos);
      sprayTrail.addLatLng(currentPos);

      const rotEl = document.getElementById('drone-icon-rot');
      if (rotEl) rotEl.style.transform = `rotate(${{calcBearing(p0, p1)}}deg)`;

      updateHUD('AUTO-SPRAYING', '#00ff66', `${{wpIdx + 1}} / ${{waypoints.length}}`, altitude, 5.0, true);
    }}

    // --- 2. RTH (TRANSIT HOME TO BASE) ---
    else if (mode === 'RTH') {{
      altitude = Math.min(5.0, altitude + 0.1);
      const heading = calcBearing(currentPos, HOME_POS);
      const rotEl = document.getElementById('drone-icon-rot');
      if (rotEl) rotEl.style.transform = `rotate(${{heading}}deg)`;

      const dLat = HOME_POS[0] - currentPos[0];
      const dLon = HOME_POS[1] - currentPos[1];
      const dist = Math.hypot(dLat, dLon);

      if (dist <= transitStep) {{
        currentPos = [...HOME_POS];
        droneMarker.setLatLng(currentPos);
        altitude = 0.0;
        tank = 30.0;
        battery = 100.0;
        mode = 'PAUSED';
        document.getElementById('homeBtn').innerText = '⚡ RESUME MISSION';
        document.getElementById('homeBtn').className = 'sim-btn';
        document.getElementById('playBtn').innerText = '▶ RESUME';
        updateHUD('LANDED & SERVICED', '#00e5ff', 'HOME BASE', 0.0, 0.0, false);
        return;
      }} else {{
        currentPos[0] += (dLat / dist) * transitStep;
        currentPos[1] += (dLon / dist) * transitStep;
        droneMarker.setLatLng(currentPos);
        updateHUD('RTH TRANSIT', '#ffaa00', 'TRANSIT -> HOME', altitude, 6.5, false);
      }}
    }}

    // --- 3. RESUME TRANSIT BACK TO BREAKPOINT ---
    else if (mode === 'RESUMING') {{
      altitude = Math.min(5.0, altitude + 0.1);
      const targetPos = breakPoint.pos;
      const heading = calcBearing(currentPos, targetPos);
      const rotEl = document.getElementById('drone-icon-rot');
      if (rotEl) rotEl.style.transform = `rotate(${{heading}}deg)`;

      const dLat = targetPos[0] - currentPos[0];
      const dLon = targetPos[1] - currentPos[1];
      const dist = Math.hypot(dLat, dLon);

      if (dist <= transitStep) {{
        currentPos = [...targetPos];
        droneMarker.setLatLng(currentPos);
        wpIdx = breakPoint.wpIdx;
        t = breakPoint.t;
        breakPoint = null;
        mode = 'SPRAYING';
        document.getElementById('homeBtn').innerText = '🏠 RETURN HOME / REFILL';
        document.getElementById('homeBtn').className = 'sim-btn btn-warn';
        document.getElementById('playBtn').innerText = '⏸ PAUSE';
      }} else {{
        currentPos[0] += (dLat / dist) * transitStep;
        currentPos[1] += (dLon / dist) * transitStep;
        droneMarker.setLatLng(currentPos);
        updateHUD('TRANSIT -> FIELD', '#00e5ff', `WP ${{breakPoint.wpIdx + 1}}`, altitude, 6.5, false);
      }}
    }}

    // --- 4. EMERGENCY LANDING (LOCAL SPOT) ---
    else if (mode === 'EMG_LAND') {{
      altitude = Math.max(0.0, altitude - 0.15);
      updateHUD('EMG LANDING', '#ff3333', 'LOCAL DESCENT', altitude, 0.0, false);
      if (altitude <= 0.0) {{
        mode = 'PAUSED';
        updateHUD('EMG LANDED (SAFE)', '#ff3333', 'MOTORS DISARMED', 0.0, 0.0, false);
        return;
      }}
    }}

    if (mode !== 'PAUSED' && mode !== 'STANDBY') {{
      animId = requestAnimationFrame(step);
    }}
  }}

  // Global Button Controls
  window.toggleFlight = function() {{
    if (mode === 'FINISHED') window.resetFlight();

    if (mode === 'SPRAYING') {{
      mode = 'PAUSED';
      document.getElementById('playBtn').innerText = '▶ RESUME';
      updateHUD('PAUSED', '#ffcc00', `${{wpIdx + 1}} / ${{waypoints.length}}`, altitude, 0.0, false);
    }} else {{
      mode = 'SPRAYING';
      document.getElementById('playBtn').innerText = '⏸ PAUSE';
      animId = requestAnimationFrame(step);
    }}
  }};

  window.triggerHome = function() {{
    if (breakPoint && mode === 'PAUSED') {{
      // Quad is landed and serviced at base: Resume back to field
      mode = 'RESUMING';
      animId = requestAnimationFrame(step);
      return;
    }}
    if (mode === 'SPRAYING' || mode === 'PAUSED') {{
      // Quad is in flight: record break position and return home
      breakPoint = {{ wpIdx: wpIdx, t: t, pos: [...currentPos] }};
      mode = 'RTH';
      animId = requestAnimationFrame(step);
    }}
  }};

  window.triggerEmgLand = function() {{
    mode = 'EMG_LAND';
    document.getElementById('playBtn').innerText = '▶ RESUME';
    animId = requestAnimationFrame(step);
  }};

  window.setSpeed = function(val) {{
    speedMult = parseFloat(val);
    document.getElementById('speedVal').innerText = parseFloat(val).toFixed(1) + 'x';
  }};

  window.resetFlight = function() {{
    mode = 'STANDBY';
    if (animId) cancelAnimationFrame(animId);
    wpIdx = 0;
    t = 0.0;
    tank = 30.0;
    battery = 100.0;
    altitude = 0.0;
    breakPoint = null;
    currentPos = [...HOME_POS];
    sprayTrail.setLatLngs([]);
    droneMarker.setLatLng(HOME_POS);
    document.getElementById('playBtn').innerText = '▶ START';
    document.getElementById('homeBtn').innerText = '🏠 RETURN HOME / REFILL';
    document.getElementById('homeBtn').className = 'sim-btn btn-warn';
    updateHUD('READY', '#ffcc00', `0 / ${{waypoints.length}}`, 0.0, 0.0, false);
  }};
</script>
</body>
</html>
"""

display(HTML(html_code))
print(f"[OK] Simulation updated: Single unified RTH & Refill action ready. {num_passes} passes across Syrdarya farm.")

[OK] Simulation updated: Single unified RTH & Refill action ready. 36 passes across Syrdarya farm.


In [7]:
from google.colab import files

# --- 1. AGRONOMIC & POWER TELEMETRY ---
SPRAY_RATE_L_HA = 18.0     # Target: 18 Liters per hectare
SPRAY_SPEED_MS = 5.0       # Cruise speed: 5 m/s
NOMAD_TANK_CAP_L = 30.0    # K30 liquid capacity
CRUISE_POWER_KW = 8.5      # Avg electrical draw under load (~157A @ 54V)
BATTERY_CAP_KWH = 1.62     # 14S 30Ah pack usable capacity (~80% DoD)

# Approximate parcel area from boundary (hectares)
def polygon_area_ha(coords):
    lat = np.radians([p[0] for p in coords[:-1]])
    lon = np.radians([p[1] for p in coords[:-1]])
    # Shoelace formula on spherical projection
    x = 6378137.0 * lon * np.cos(np.mean(lat))
    y = 6378137.0 * lat
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1))) / 10000.0

field_area_ha = polygon_area_ha(field_boundary)
total_spray_vol_l = field_area_ha * SPRAY_RATE_L_HA
flight_time_min = (mission_distance_m / SPRAY_SPEED_MS) / 60.0
energy_used_kwh = (CRUISE_POWER_KW * (flight_time_min / 60.0))
batteries_required = int(np.ceil(energy_used_kwh / BATTERY_CAP_KWH))
tanks_required = int(np.ceil(total_spray_vol_l / NOMAD_TANK_CAP_L))

print("==================================================")
print("       NOMAD K30 MISSION TELEMETRY DASHBOARD      ")
print("==================================================")
print(f"• Parcel Area:          {field_area_ha:.2f} ha")
print(f"• Total Track Distance: {mission_distance_m/1000.0:.2f} km ({num_passes} passes)")
print(f"• Flight Time (5 m/s):  {flight_time_min:.1f} minutes")
print(f"• Required Chemical:    {total_spray_vol_l:.1f} Liters ({tanks_required} tank fill(s))")
print(f"• Energy Consumed:      {energy_used_kwh:.2f} kWh (~{batteries_required} battery cycle(s))")
print("==================================================")

# --- 2. GENERATE ARDUPILOT QGC WPL 110 FILE ---
# Format: <INDEX> <CURRENT_WP> <COORD_FRAME> <COMMAND> <P1> <P2> <P3> <P4> <LAT> <LON> <ALT> <AUTOCONTINUE>
wp_lines = ["QGC WPL 110"]

# Waypoint 0: Home Position (Ground Level)
wp_lines.append(f"0\t1\t0\t16\t0\t0\t0\t0\t{P_NW[0]:.8f}\t{P_NW[1]:.8f}\t0.0\t1")

# Waypoint 1: Autonomous Takeoff to 3.0 meters AGL
wp_lines.append(f"1\t0\t3\t22\t0\t0\t0\t0\t{P_NW[0]:.8f}\t{P_NW[1]:.8f}\t3.0\t1")

# Swath Waypoints: 3.0 meters spraying altitude
wp_idx = 2
for pt in flight_path:
    wp_lines.append(f"{wp_idx}\t0\t3\t16\t0\t0\t0\t0\t{pt[0]:.8f}\t{pt[1]:.8f}\t3.0\t1")
    wp_idx += 1

# Final Waypoint: Return-to-Launch (RTL)
wp_lines.append(f"{wp_idx}\t0\t3\t20\t0\t0\t0\t0\t0.0\t0.0\t0.0\t1")

filename = "nomad_k30_syrdarya_mission.waypoints"
with open(filename, "w") as f:
    f.write("\n".join(wp_lines))

print(f"\n[OK] Generated {wp_idx} ArduPilot waypoints.")
print(f"Downloading '{filename}'...")
files.download(filename)

       NOMAD K30 MISSION TELEMETRY DASHBOARD      
• Parcel Area:          3.77 ha
• Total Track Distance: 7.33 km (36 passes)
• Flight Time (5 m/s):  24.4 minutes
• Required Chemical:    67.9 Liters (3 tank fill(s))
• Energy Consumed:      3.46 kWh (~3 battery cycle(s))

[OK] Generated 76 ArduPilot waypoints.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Duszed/Nomad-Aerospace-Flight-Systems/blob/main/notebooks/mission_planner_demo.ipynb)

SyntaxError: invalid syntax (2314532727.py, line 1)